In [1]:
import itertools
import os
import random
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg as sla
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

SEED = 42
TARGET = 0.90
NS = (20, 30, 40)
FRACS = (0.25, 0.5, 1.0)
COLORS = {20: "#4878d0", 30: "#ee854a", 40: "#6acc64"}
WEIGHT_DIR = "./weights/baseline_fs"


def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")

## Feature selection
Each step adds the single feature or the pair of features that most improves validation accuracy. Pairs let the
search pick up XOR-style interactions that a single bit cannot show. Only the selected columns are passed on to the
baseline.

In [2]:
def lookup_table_acc(S_train, y_train, S_val, y_val, cols):
    """Val accuracy of a majority-vote lookup table over the bits in `cols`."""
    w = 2 ** np.arange(len(cols))
    idx_tr = ((S_train[:, cols] > 0) * w).sum(1)
    idx_va = ((S_val[:, cols] > 0) * w).sum(1)
    count = np.bincount(idx_tr, minlength=2 ** len(cols))
    pos = np.bincount(idx_tr, weights=y_train, minlength=2 ** len(cols))
    table = (2 * pos > count).astype(int)  # unseen cells default to 0
    return (table[idx_va] == y_val).mean()


def select_features(S_train, y_train, S_val, y_val, min_gain=0.005):
    """Greedy forward selection that may add one bit or a pair of bits per step."""
    n = S_train.shape[1]
    score = lambda cols: lookup_table_acc(S_train, y_train, S_val, y_val, list(cols))
    selected, best = [], 0.5
    while True:
        rest = [j for j in range(n) if j not in selected]
        cands = [[j] for j in rest] + [list(p) for p in itertools.combinations(rest, 2)]
        acc, add = max((score(selected + c), c) for c in cands)
        if acc <= best + min_gain:
            break
        selected, best = selected + add, acc
        print(f"  + {add} -> {sorted(selected)}  val acc {acc:.4f}")
    return sorted(selected)


def mask_features(X, cols):
    """Zero out every feature not in `cols`, so KryptoNet still receives 40 inputs."""
    M = np.zeros_like(X)
    M[:, cols] = X[:, cols]
    return M


## Baseline code

In [3]:
def set_seed(seed=42):
    """Set a seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_tensors(X, y):
    """Convert numpy arrays to float tensors."""
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()


def accuracy(net, X, y):
    """Get accuracy of `net` on `X` and `y`."""
    net.eval()
    with torch.no_grad():
        return float(((net(X).squeeze(1) > 0).float() == y).float().mean())

def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test

def poly_logreg(X_train, y_train, X_test, y_test, degree):
    """Logistic regression on explicit polynomial features."""
    poly = PolynomialFeatures(degree)
    logreg = LogisticRegression(max_iter=100, solver="sag", C=0.85)
    logreg.fit(poly.fit_transform(X_train), y_train)
    return float(logreg.score(poly.transform(X_test), y_test))

def agreements(X1, X2, n):
    """Number of agreeing bits between every pair of rows, after rounding the inputs to bits."""
    B1, B2 = np.sign(X1).astype(np.float32), np.sign(X2).astype(np.float32)
    return np.rint((n + B1 @ B2.T) / 2).astype(np.uint8)


def polyfeatures_kernel_table(n, degree):
    """phi(x).phi(x') for PolynomialFeatures(degree) on +-1 bits, indexed by the number m of agreeing bits."""
    table = np.zeros(n + 1)
    for s in range(degree + 1):
        weight = comb(n + (degree - s) // 2, n)
        e_s = [sum(comb(m, a) * comb(n - m, s - a) * (-1) ** (s - a) for a in range(s + 1)) for m in range(n + 1)]
        table += weight * np.array(e_s, dtype=float)
    return table / table[n]


def poly_kernel(X1, X2, degree, block=256):
    """phi(x).phi(x') for PolynomialFeatures(degree) on raw (not binarised) inputs, divided by the number of features.

    With z_i = x_i * x'_i this is sum_{k <= degree} h_k(z), where h_k is the complete homogeneous symmetric polynomial.
    h_k follows from the power sums p_j(z) = (X1**j) @ (X2**j).T by Newton's identities. On +-1 inputs it equals
    polyfeatures_kernel_table(n, degree)[agreements(X1, X2, n)].
    """
    n = X1.shape[1]
    P1 = [np.asarray(X1, np.float64) ** j for j in range(1, degree + 1)]
    P2 = [np.asarray(X2, np.float64).T ** j for j in range(1, degree + 1)]
    K = np.empty((len(X1), len(X2)), np.float32)
    for s in range(0, len(X1), block):
        p = [A[s:s + block] @ B for A, B in zip(P1, P2)]
        h = [np.ones_like(p[0])]
        for k in range(1, degree + 1):
            h.append(sum(p[j - 1] * h[k - j] for j in range(1, k + 1)) / k)
        K[s:s + block] = sum(h) / comb(n + degree, degree)
    return K


def kernel_logreg(K, y, C=0.85, iters=30, tol=1e-6):
    """C * sum(log-loss) + ||w||^2 / 2 in the dual, by Newton's method (Rasmussen & Williams, Alg. 3.1)."""
    t = y.astype(np.float64); sign = 2 * t - 1
    a, f = np.zeros(len(t), np.float32), np.zeros(len(t))
    obj = C * len(t) * np.log(2)
    for _ in range(iters):
        pi = 1 / (1 + np.exp(-f))
        W = C * pi * (1 - pi); sW = np.sqrt(W).astype(np.float32)
        B = K * sW[:, None]; B *= sW[None, :]; B[np.diag_indices_from(B)] += 1
        L = sla.cholesky(B, lower=True, overwrite_a=True, check_finite=False)
        del B
        b = (W * f + C * (t - pi)).astype(np.float32)
        v = sla.solve_triangular(L, sW * (K @ b), lower=True, check_finite=False)
        a_new = b - sW * sla.solve_triangular(L, v, lower=True, trans="T", check_finite=False)
        del L
        f_new = (K @ a_new).astype(np.float64)
        new = C * np.logaddexp(0, -sign * f_new).sum() + 0.5 * float(a_new @ f_new)
        for _ in range(5):
            if new <= obj:
                break
            a_new, f_new = (a + a_new) / 2, (f + f_new) / 2
            new = C * np.logaddexp(0, -sign * f_new).sum() + 0.5 * float(a_new @ f_new)
        done = abs(obj - new) < tol * abs(new)
        a, f, obj = a_new, f_new, new
        if done:
            break
    return a


## Feature selection → baseline on 25%, 50%, 100% of the data

In [4]:
DEGREES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
CS = (0.85, 10, 100, 1000)


def kernel_predict(S_query, S_support, a, degree):
    """Labels from the dual weights `a` of `kernel_logreg`, using only the selected columns."""
    k = S_support.shape[1]
    table = polyfeatures_kernel_table(k, degree).astype(np.float32)
    return (table[agreements(S_query, S_support, k)] @ a > 0).astype(int)


def baseline_predict(S_query, S_support, a, degree, binary=True):
    """Labels from the dual weights `a`: the kernel on signs (`binary`), or the same kernel on raw values."""
    if binary:
        return kernel_predict(S_query, S_support, a, degree)
    return (poly_kernel(S_query, S_support, degree) @ a > 0).astype(int)


def run(n, frac, fs=True, binary=True, weight_dir=WEIGHT_DIR):
    """Feature selection (`fs`, else all features), then a degree x C search for the baseline on validation, then
    one test score. `binary` uses the sign of each feature; otherwise the kernel is computed on the raw values."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    Xa, ya, Xv, yv = X_train[:m], y_train[:m], X_val[:mv], y_val[:mv]
    print(f"=== n={n}, {frac:.0%}: {m} train / {mv} val / {len(y_test)} test ===")

    cols = select_features(Xa, ya, Xv, yv) if fs else list(range(n))
    k = len(cols)
    Sa, Sv, St = Xa[:, cols], Xv[:, cols], X_test[:, cols]
    if binary:
        M_train, M_val = agreements(Sa, Sa, k), agreements(Sv, Sa, k)

    best = None
    for d in DEGREES:
        if binary:
            table = polyfeatures_kernel_table(k, d).astype(np.float32)
            K, K_val = table[M_train], table[M_val]
        else:
            K, K_val = poly_kernel(Sa, Sa, d), poly_kernel(Sv, Sa, d)
        for C in CS:
            a = kernel_logreg(K, ya, C=C)
            val = float(((K_val @ a > 0).astype(int) == yv).mean())
            if best is None or val > best["val"]:
                best = {"degree": d, "C": C, "val": val, "a": a}
        print(f"  degree={d:>2} | best so far: val {best['val']:.4f} (degree={best['degree']}, C={best['C']})")
        del K, K_val
    M_train = M_val = None

    test = float((baseline_predict(St, Sa, best["a"], best["degree"], binary) == y_test).mean())
    out_dir = os.path.join(weight_dir, f"n{n}", f"{round(frac * 100)}pct")
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "model.npz")
    np.savez(path, a=best["a"], support=Sa.astype(np.float32), features=np.array(cols),
             degree=best["degree"], C=best["C"], binarise=binary)
    print(f"  {'selected' if fs else 'all'} {k} features {cols} | degree {best['degree']}, C {best['C']} | "
          f"val {best['val']:.4f} | test {test:.4f} -> {path}")
    return {"n": n, "frac": frac, "train_size": m, "features": cols, "degree": best["degree"], "C": best["C"],
            "val": best["val"], "test": test, "path": path, "binary": binary}


results = {}
for n in NS:
    for frac in FRACS:
        results[n, frac] = run(n, frac)

=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5550
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6117
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6583
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7350
  + [0, 2] -> [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]  val acc 0.8700
  degree= 1 | best so far: val 0.5150 (degree=1, C=0.85)
  degree= 2 | best so far: val 0.5283 (degree=2, C=100)
  degree= 3 | best so far: val 0.5850 (degree=3, C=1000)
  degree= 4 | best so far: val 0.6700 (degree=4, C=1000)
  degree= 5 | best so far: val 0.7950 (degree=5, C=1000)
  degree= 6 | best so far: val 0.8250 (degree=6, C=1000)
  degree= 7 | best so far: val 0.8250 (degree=6, C=1000)
  degree= 8 | best so far: val 0.8250 (degree=6, C=1000)
  degree= 9 | best so far: val 0.8250 (degree=6, C=1000)
  degree=10 | best so far: val 0.8250 (degree=6, C=1000)
  selected 10 features [0, 1, 2, 4, 5, 7, 9, 10, 12, 15] | degree 6, C 1000 | val 0.8250 | test 0.8550 -> ./weights/base

## Summary

In [5]:
print(f"{'n':>3} | {'subset':>6} | {'train':>6} | {'#feat':>5} | {'degree':>6} | {'C':>6} | {'val':>6} | {'test':>6}")
for (n, frac), r in results.items():
    flag = "" if r["test"] >= TARGET else "  < target"
    print(f"{n:>3} | {frac:>6.0%} | {r['train_size']:>6} | {len(r['features']):>5} | {r['degree']:>6} | "
          f"{r['C']:>6g} | {r['val']:.4f} | {r['test']:.4f}{flag}")

  n | subset |  train | #feat | degree |      C |    val |   test
 20 |    25% |   1800 |    10 |      6 |   1000 | 0.8250 | 0.8550  < target
 20 |    50% |   3600 |    10 |     10 |   1000 | 0.9375 | 0.9404
 20 |   100% |   7200 |    10 |      8 |   1000 | 0.9533 | 0.9592
 30 |    25% |   3000 |    10 |      6 |   1000 | 0.9080 | 0.9080
 30 |    50% |   6000 |    10 |     10 |   1000 | 0.9550 | 0.9565
 30 |   100% |  12000 |    10 |      7 |   1000 | 0.9565 | 0.9580
 40 |    25% |   4800 |    10 |      8 |   1000 | 0.9381 | 0.9569
 40 |    50% |   9600 |    10 |      7 |   1000 | 0.9556 | 0.9623
 40 |   100% |  19200 |    10 |      6 |   1000 | 0.9569 | 0.9631


## Variants without feature selection and/or binarisation
The same degree x C search, split and fractions as above, in three more settings:

- **No feature selection:** the baseline uses all n features.
- **No binarisation:** the kernel is computed on the raw feature values instead of their signs. It is still the exact
  inner product of `PolynomialFeatures(degree)` (see `poly_kernel`), so this is the polynomial logistic regression of
  the paper on the raw inputs, solved in the dual.

Each setting saves its best model per (n, fraction) to its own folder in the same `model.npz` layout.

In [6]:
VARIANTS = {
    "FS, no binarise": {"fs": True, "binary": False, "weight_dir": "./weights/baseline_fs_no_bin"},
    "No FS, binarise": {"fs": False, "binary": True, "weight_dir": "./weights/baseline_no_fs"},
    "No FS, no binarise": {"fs": False, "binary": False, "weight_dir": "./weights/baseline_no_fs_no_bin"},
}
variant_results = {"FS, binarise": results}


def run_variant(name):
    """Run every (n, fraction) for one variant."""
    variant_results[name] = {(n, frac): run(n, frac, **VARIANTS[name]) for n in NS for frac in FRACS}

### Feature selection, no binarisation

In [7]:
run_variant("FS, no binarise")

=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5550
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6117
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6583
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7350
  + [0, 2] -> [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]  val acc 0.8700
  degree= 1 | best so far: val 0.5217 (degree=1, C=1000)
  degree= 2 | best so far: val 0.5367 (degree=2, C=1000)
  degree= 3 | best so far: val 0.5983 (degree=3, C=10)
  degree= 4 | best so far: val 0.6400 (degree=4, C=100)
  degree= 5 | best so far: val 0.7150 (degree=5, C=100)
  degree= 6 | best so far: val 0.7533 (degree=6, C=100)
  degree= 7 | best so far: val 0.7583 (degree=7, C=100)
  degree= 8 | best so far: val 0.7817 (degree=8, C=1000)
  degree= 9 | best so far: val 0.7817 (degree=8, C=1000)
  degree=10 | best so far: val 0.7900 (degree=10, C=100)
  selected 10 features [0, 1, 2, 4, 5, 7, 9, 10, 12, 15] | degree 10, C 100 | val 0.7900 | test 0.8079 -> ./weights/baseline_

### No feature selection, binarisation

In [8]:
run_variant("No FS, binarise")

=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  degree= 1 | best so far: val 0.4950 (degree=1, C=1000)
  degree= 2 | best so far: val 0.5383 (degree=2, C=0.85)
  degree= 3 | best so far: val 0.5383 (degree=2, C=0.85)
  degree= 4 | best so far: val 0.5383 (degree=2, C=0.85)
  degree= 5 | best so far: val 0.5433 (degree=5, C=100)
  degree= 6 | best so far: val 0.5517 (degree=6, C=100)
  degree= 7 | best so far: val 0.5517 (degree=6, C=100)
  degree= 8 | best so far: val 0.5517 (degree=6, C=100)
  degree= 9 | best so far: val 0.5517 (degree=6, C=100)
  degree=10 | best so far: val 0.5517 (degree=6, C=100)
  all 20 features [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19] | degree 6, C 100 | val 0.5517 | test 0.5504 -> ./weights/baseline_no_fs\n20\25pct\model.npz
=== n=20, 50%: 3600 train / 1200 val / 2400 test ===
  degree= 1 | best so far: val 0.4883 (degree=1, C=100)
  degree= 2 | best so far: val 0.5500 (degree=2, C=0.85)
  degree= 3 | best so far: val 0.550

### No feature selection, no binarisation

In [9]:
run_variant("No FS, no binarise")

=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  degree= 1 | best so far: val 0.5017 (degree=1, C=0.85)
  degree= 2 | best so far: val 0.5417 (degree=2, C=0.85)
  degree= 3 | best so far: val 0.5417 (degree=2, C=0.85)
  degree= 4 | best so far: val 0.5417 (degree=2, C=0.85)
  degree= 5 | best so far: val 0.5417 (degree=2, C=0.85)
  degree= 6 | best so far: val 0.5417 (degree=2, C=0.85)
  degree= 7 | best so far: val 0.5417 (degree=2, C=0.85)
  degree= 8 | best so far: val 0.5417 (degree=2, C=0.85)
  degree= 9 | best so far: val 0.5417 (degree=2, C=0.85)
  degree=10 | best so far: val 0.5550 (degree=10, C=0.85)
  all 20 features [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19] | degree 10, C 0.85 | val 0.5550 | test 0.5492 -> ./weights/baseline_no_fs_no_bin\n20\25pct\model.npz
=== n=20, 50%: 3600 train / 1200 val / 2400 test ===
  degree= 1 | best so far: val 0.4958 (degree=1, C=0.85)
  degree= 2 | best so far: val 0.5558 (degree=2, C=10)
  degree= 3 | best so

### Summary of all variants

In [10]:
names = list(variant_results)
print(f"{'':>3} | {'':>6} | " + " | ".join(f"{name:^19}" for name in names))
print(f"{'n':>3} | {'subset':>6} | " + " | ".join(f"{'val':>9} {'test':>9}" for _ in names))
for n in NS:
    for frac in FRACS:
        rows = [variant_results[name][n, frac] for name in names]
        print(f"{n:>3} | {frac:>6.0%} | " + " | ".join(f"{r['val']:>9.4f} {r['test']:>9.4f}" for r in rows))

    |        |    FS, binarise     |   FS, no binarise   |   No FS, binarise   | No FS, no binarise 
  n | subset |       val      test |       val      test |       val      test |       val      test
 20 |    25% |    0.8250    0.8550 |    0.7900    0.8079 |    0.5517    0.5504 |    0.5550    0.5492
 20 |    50% |    0.9375    0.9404 |    0.8617    0.8679 |    0.5750    0.5583 |    0.5583    0.5337
 20 |   100% |    0.9533    0.9592 |    0.9154    0.9221 |    0.5829    0.6008 |    0.5725    0.5754
 30 |    25% |    0.9080    0.9080 |    0.8540    0.8518 |    0.5570    0.5200 |    0.5580    0.5120
 30 |    50% |    0.9550    0.9565 |    0.9030    0.8918 |    0.5505    0.5440 |    0.5555    0.5475
 30 |   100% |    0.9565    0.9580 |    0.9363    0.9353 |    0.5590    0.5593 |    0.5605    0.5615
 40 |    25% |    0.9381    0.9569 |    0.8762    0.8922 |    0.5350    0.5119 |    0.5381    0.5127
 40 |    50% |    0.9556    0.9623 |    0.9197    0.9266 |    0.5353    0.5134 |    0.5316 

## Reload saved weights and check test accuracy

In [11]:
for name, res in variant_results.items():
    for (n, frac), r in res.items():
        w = np.load(r["path"])
        cols = w["features"].tolist()
        _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{n}"))
        pred = baseline_predict(X_test[:, cols], w["support"], w["a"], int(w["degree"]), bool(w["binarise"]))
        acc = float((pred == y_test).mean())
        assert abs(acc - r["test"]) < 1e-9, (name, n, frac, acc, r["test"])
        print(f"{name:<18} n={n} {frac:>4.0%}: reloaded {r['path']} -> test {acc:.4f}")

FS, binarise       n=20  25%: reloaded ./weights/baseline_fs\n20\25pct\model.npz -> test 0.8550
FS, binarise       n=20  50%: reloaded ./weights/baseline_fs\n20\50pct\model.npz -> test 0.9404
FS, binarise       n=20 100%: reloaded ./weights/baseline_fs\n20\100pct\model.npz -> test 0.9592
FS, binarise       n=30  25%: reloaded ./weights/baseline_fs\n30\25pct\model.npz -> test 0.9080
FS, binarise       n=30  50%: reloaded ./weights/baseline_fs\n30\50pct\model.npz -> test 0.9565
FS, binarise       n=30 100%: reloaded ./weights/baseline_fs\n30\100pct\model.npz -> test 0.9580
FS, binarise       n=40  25%: reloaded ./weights/baseline_fs\n40\25pct\model.npz -> test 0.9569
FS, binarise       n=40  50%: reloaded ./weights/baseline_fs\n40\50pct\model.npz -> test 0.9623
FS, binarise       n=40 100%: reloaded ./weights/baseline_fs\n40\100pct\model.npz -> test 0.9631
FS, no binarise    n=20  25%: reloaded ./weights/baseline_fs_no_bin\n20\25pct\model.npz -> test 0.8079
FS, no binarise    n=20  50%: 